**Goals:**
- Interactivity --> see details
- Map routes
- Filtering --> ex// Show only high deficit routes
- Narrative --> ex// This area shows consistent serice gaps

**Do:**
- Simple map
- Clear coloring
- Clickable insights

**Add:**
- filter by years like 2026 - 2028 (so we have the forecasted values)
- integrate ACS and time series
- what if can be a side by side (current vs. projection such as if we change bus stops and route frequency so we have snapshots to see side by side scenarios)

In [ ]:
# ── 0. Install & Imports ──────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['gradio', 'geopandas', 'plotly', 'joblib', 'xgboost', 'requests']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import gradio as gr
import pandas as pd
import numpy as np
import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go
import joblib
import requests, json, os
import warnings
warnings.filterwarnings('ignore')

In [ ]:

# ── 1. Load Data ──────────────────────────────────────────────────────────────
# Loads the features CSV you already have from Sprint 2b.
# Change this path if your CSV is named differently.
CSV_PATH = "Sprint2b_Modeling_Features_NotebookOutput.csv"

df = pd.read_csv(CSV_PATH)

# If the model .pkl is available, load it. Otherwise we use the raw
# composite_access_deficit column as a stand-in so the map still works
# while the model is still being tuned.
MODEL_PATH  = "Sprint2b_XGBoost_v3.pkl"   # or "model.pkl"
SCALER_PATH = "Sprint2b_Scaler_v3.pkl"    # or "scaler.pkl"

FEATURE_NAMES = [
    'headway_early_min', 'headway_peak_am_min',
    'freq_early_tph', 'freq_peak_am_tph',
    'weekend_weekday_ratio', 'rail_trip_share',
    'neighbor_mean_equity_score', 'neighbor_mean_headway_peak', 'n_neighbors',
    'trend_commute_public_transit_pct', 'trend_commute_drove_alone_pct',
    'trend_commute_wfh_pct', 'trend_mean_commute_time_min',
]

model, scaler = None, None
if os.path.exists(MODEL_PATH):
    model  = joblib.load(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH) if os.path.exists(SCALER_PATH) else None
    print("✅ Model loaded")
else:
    print("⚠️  model.pkl not found — using composite_access_deficit as proxy score")

⚠️  model.pkl not found — using composite_access_deficit as proxy score


In [ ]:
# ── 2. Build deficit score column ─────────────────────────────────────────────
# If model is loaded, predict for every tract. Otherwise use the existing column.
if model is not None:
    X = df[FEATURE_NAMES].fillna(0)
    df['deficit_predicted'] = model.predict(X)
else:
    # Fallback: normalize composite_access_deficit to [0, 1]
    col = df['composite_access_deficit']
    df['deficit_predicted'] = (col - col.min()) / (col.max() - col.min())

# Risk tier from predicted score
df['risk_tier'] = pd.qcut(
    df['deficit_predicted'], 4,
    labels=['Low', 'Moderate', 'High', 'Critical']
)


In [ ]:
# ── 3. Fetch Miami-Dade Census Tract GeoJSON ──────────────────────────────────
# This downloads automatically — no manual file needed.
# Uses the Census Bureau's free TIGER/Web API for Miami-Dade (FIPS county = 086).
# TIGER provides geometry — the shape/boundary of each tract

GEOJSON_URL = (
    "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
    "tigerWMS_ACS2023/MapServer/8/query"
    "?where=STATE%3D'12'+AND+COUNTY%3D'086'"     #filters to state florida and county Miami-Dade
    "&outFields=GEOID,NAME&f=geojson&outSR=4326"  #returns the data in a JSON and use long. and lat.
)
GEOJSON_CACHE = "miami_dade_tracts.geojson"

def load_geodata():
    if os.path.exists(GEOJSON_CACHE):
        print("✅ GeoJSON loaded from cache")
        return gpd.read_file(GEOJSON_CACHE)
    print("📡 Downloading Miami-Dade tract geometries from Census Bureau...")
    r = requests.get(GEOJSON_URL, timeout=30)
    with open(GEOJSON_CACHE, 'w') as f:
        json.dump(r.json(), f)
    gdf = gpd.read_file(GEOJSON_CACHE)
    print(f"✅ Downloaded {len(gdf)} tracts")
    return gdf

gdf_raw = load_geodata()

# Align GEOIDs: Census uses 11-digit strings like "12086XXXXXXX"
# Your tract_geoid is numeric — zero-pad it to 11 digits to match
df['geoid_str'] = df['tract_geoid'].astype(str).str.zfill(11)
gdf_raw['GEOID'] = gdf_raw['GEOID'].astype(str).str.zfill(11)

# Merge geometry with your data
gdf = gdf_raw.merge(df, left_on='GEOID', right_on='geoid_str', how='inner')
print(f"✅ Merged: {len(gdf)} tracts with geometry + deficit scores")

# Convert to WGS84 and compute centroids for hover labels
gdf = gdf.to_crs(epsg=4326)

✅ GeoJSON loaded from cache
✅ Merged: 504 tracts with geometry + deficit scores


In [ ]:
# ── 4. Map builder ────────────────────────────────────────────────────────────
TIER_ORDER  = ['Low', 'Moderate', 'High', 'Critical']
TIER_COLORS = {
    'Low':      '#27ae60',   # green
    'Moderate': '#f39c12',   # amber
    'High':     '#e67e22',   # orange
    'Critical': '#c0392b',   # red
}

def build_map(selected_tiers, score_threshold):
    """Rebuild the choropleth map based on active filters."""
    subset = gdf[
        gdf['risk_tier'].isin(selected_tiers) &
        (gdf['deficit_predicted'] >= score_threshold)
    ].copy()

    if subset.empty:
        fig = go.Figure()
        fig.update_layout(
            title="No tracts match current filters",
            paper_bgcolor='rgba(0,0,0,0)',
            plot_bgcolor='rgba(0,0,0,0)'
        )
        return fig

    # Build a GeoJSON dict for Plotly
    geojson_dict = json.loads(subset.to_json())

    fig = px.choropleth_mapbox(
        subset,
        geojson=geojson_dict,
        locations=subset.index,
        color='deficit_predicted',
        color_continuous_scale=[
            [0.0,  '#27ae60'],
            [0.33, '#f1c40f'],
            [0.66, '#e67e22'],
            [1.0,  '#c0392b'],
        ],
        range_color=[0, 1],
        mapbox_style='open-street-map',
        zoom=9,
        center={'lat': 25.77, 'lon': -80.19},
        opacity=0.65,
        hover_data={
            'deficit_predicted': ':.3f',
            'risk_tier': True,
            'poverty_rate_pct': ':.1f',
            'hh_no_vehicle_pct': ':.1f',
            'stop_count': ':.0f',
        },
        labels={
            'deficit_predicted': 'Deficit Score',
            'risk_tier': 'Risk Tier',
            'poverty_rate_pct': 'Poverty Rate (%)',
            'hh_no_vehicle_pct': 'No Vehicle HH (%)',
            'stop_count': 'Bus Stops',
        },
    )

    fig.update_layout(
        margin=dict(l=0, r=0, t=30, b=0),
        height=550,
        coloraxis_colorbar=dict(
            title='Deficit',
            tickvals=[0, 0.25, 0.5, 0.75, 1.0],
            ticktext=['0 (Best)', '0.25', '0.50', '0.75', '1 (Worst)'],
            len=0.6,
        ),
        title=dict(
            text=f"Miami-Dade Transit Deficit Map  •  {len(subset)} tracts shown",
            font=dict(size=14)
        )
    )
    return fig

In [ ]:
# ── 5. Narrative generator ────────────────────────────────────────────────────
def generate_narrative(tract_id_input):
    """
    Given a tract GEOID (or numeric ID), return a plain-English summary
    of that tract's transit situation.
    """
    if not tract_id_input:
        return "Enter a tract ID above to see its detailed narrative."

    # Try matching by numeric or string geoid
    try:
        tid = str(int(float(tract_id_input))).zfill(11)
    except:
        tid = str(tract_id_input).strip().zfill(11)

    match = gdf[gdf['GEOID'] == tid]
    if match.empty:
        # Try partial match
        match = gdf[gdf['GEOID'].str.contains(tract_id_input.strip())]
    if match.empty:
        return f"❌ Tract '{tract_id_input}' not found. Try a full 11-digit GEOID like 12086010100."

    row = match.iloc[0]
    tier  = row['risk_tier']
    score = row['deficit_predicted']

    # Identify the top 3 driver flags
    drivers = []
    if pd.notna(row.get('headway_peak_am_min')) and row['headway_peak_am_min'] > 30:
        drivers.append(f"long peak headways ({row['headway_peak_am_min']:.0f} min between buses)")
    if pd.notna(row.get('weekend_weekday_ratio')) and row['weekend_weekday_ratio'] < 0.5:
        drivers.append(f"very limited weekend service (ratio: {row['weekend_weekday_ratio']:.2f})")
    if pd.notna(row.get('freq_peak_am_tph')) and row['freq_peak_am_tph'] < 2:
        drivers.append(f"low peak AM frequency ({row['freq_peak_am_tph']:.1f} trips/hr)")
    if pd.notna(row.get('stop_count')) and row['stop_count'] < 5:
        drivers.append(f"very few bus stops ({row['stop_count']:.0f})")
    if pd.notna(row.get('hh_no_vehicle_pct')) and row['hh_no_vehicle_pct'] > 20:
        drivers.append(f"high car-free household rate ({row['hh_no_vehicle_pct']:.1f}%)")

    tier_emoji = {'Low': '🟢', 'Moderate': '🟡', 'High': '🟠', 'Critical': '🔴'}.get(str(tier), '⚪')

    lines = [
        f"**Tract {tract_id_input}**",
        f"{tier_emoji} **Risk Tier: {tier}**  |  Deficit Score: {score:.3f}",
        "",
        "**Demographics:**",
        f"• Poverty rate: {row.get('poverty_rate_pct', 'N/A'):.1f}%",
        f"• Households with no vehicle: {row.get('hh_no_vehicle_pct', 'N/A'):.1f}%",
        f"• SNAP recipients: {row.get('snap_benefits_pct', 'N/A'):.1f}%",
        "",
        "**Transit Service:**",
        f"• Bus stops: {row.get('stop_count', 'N/A'):.0f}",
        f"• Routes: {row.get('route_count', 'N/A'):.0f}",
        f"• Peak AM headway: {row.get('headway_peak_am_min', 'N/A'):.0f} min",
        f"• Weekend/weekday ratio: {row.get('weekend_weekday_ratio', 'N/A'):.2f}",
        f"• Service span: {row.get('mean_service_span_hours', 'N/A'):.1f} hrs/day",
        "",
    ]

    if drivers:
        lines.append("**Primary gap drivers:**")
        for d in drivers[:3]:
            lines.append(f"• {d}")
        lines.append("")

    # Narrative sentence
    if str(tier) == 'Critical':
        lines.append(
            f"⚠️ This tract shows **consistent, compounding service gaps**. "
            f"High car dependency, limited frequencies, and poor weekend coverage "
            f"all reinforce each other. Targeted headway reduction would have the "
            f"highest impact here."
        )
    elif str(tier) == 'High':
        lines.append(
            f"This tract has **above-average transit deficits**. "
            f"Service improvements — particularly around frequency and span — "
            f"could meaningfully reduce transit burden for residents."
        )
    elif str(tier) == 'Moderate':
        lines.append(
            f"This tract has **moderate service gaps**, likely concentrated "
            f"in off-peak or weekend windows. Targeted schedule adjustments "
            f"could close the gap efficiently."
        )
    else:
        lines.append(
            f"This tract has **relatively good transit coverage** "
            f"compared to the rest of Miami-Dade."
        )

    return "\n".join(lines)


In [ ]:
# ── 6. What-If Simulator ──────────────────────────────────────────────────────
def whatif_simulate(headway_peak, freq_peak, weekend_ratio, rail_share):
    """
    Takes slider inputs for the 4 most important features (per SHAP),
    fills in median values for the other 9, and predicts deficit.
    Also returns a comparison to the current median.
    """
    if model is None:
        return "⚠️ Model not loaded — upload model.pkl to enable this feature."

    medians = df[FEATURE_NAMES].median()
    X_input = medians.copy()
    X_input['headway_peak_am_min']  = headway_peak
    X_input['freq_peak_am_tph']     = freq_peak
    X_input['weekend_weekday_ratio'] = weekend_ratio
    X_input['rail_trip_share']      = rail_share

    pred = model.predict(X_input.values.reshape(1, -1))[0]
    baseline = model.predict(medians.values.reshape(1, -1))[0]
    delta = pred - baseline

    direction = "↑ worse" if delta > 0 else "↓ better"
    color_flag = "🔴" if pred > 0.65 else "🟠" if pred > 0.45 else "🟢"

    return (
        f"{color_flag} **Predicted Deficit: {pred:.3f}**\n\n"
        f"vs. Miami-Dade median ({baseline:.3f}): "
        f"{'**+' if delta >= 0 else '**'}{delta:.3f}** {direction}\n\n"
        f"{'⚠️ This configuration produces Critical-tier service.' if pred > 0.65 else ''}"
        f"{'✅ This configuration is near or below the median deficit.' if pred < 0.45 else ''}"
    )


In [ ]:
# ── 7. Summary Stats ──────────────────────────────────────────────────────────
def summary_stats():
    counts = df['risk_tier'].value_counts()
    lines = ["**Miami-Dade Transit Deficit — Overview**\n"]
    lines.append(f"Total census tracts analyzed: **{len(df)}**\n")
    for tier in ['Critical', 'High', 'Moderate', 'Low']:
        n = counts.get(tier, 0)
        pct = 100 * n / len(df)
        emoji = {'Critical': '🔴', 'High': '🟠', 'Moderate': '🟡', 'Low': '🟢'}[tier]
        lines.append(f"{emoji} **{tier}**: {n} tracts ({pct:.0f}%)")
    lines.append(f"\nModel: XGBoost  |  Test R²: 0.808  |  CV R²: 0.821 ± 0.022")
    return "\n".join(lines)

In [ ]:

# ── 8. Build Gradio App ───────────────────────────────────────────────────────
INITIAL_TIERS = ['Low', 'Moderate', 'High', 'Critical']

with gr.Blocks(
    title="Miami-Dade Transit Deficit Dashboard",
    theme=gr.themes.Soft(),
    css=".gradio-container { max-width: 1200px !important }"
) as demo:

    # Header
    gr.Markdown("""
    # 🚌 Miami-Dade Transit Service Deficit Dashboard
    **University of Miami — AI for Equitable Public Transportation Deloitte Capstone**
    Predicting and mapping transit service gaps across Miami-Dade census tracts.
    """)

    # Summary bar
    gr.Markdown(summary_stats())

    # ── Tab 1: Map + Filter ──────────────────────────────────────────────────
    with gr.Tab("🗺️ Deficit Map"):
        gr.Markdown("Use the filters below to isolate tracts by risk tier and minimum deficit score.")

        with gr.Row():
            tier_filter = gr.CheckboxGroup(
                choices=TIER_ORDER,
                value=INITIAL_TIERS,
                label="Show Risk Tiers"
            )
            score_slider = gr.Slider(
                minimum=0.0, maximum=1.0, value=0.0, step=0.05,
                label="Minimum Deficit Score (drag right to focus on worst tracts)"
            )

        map_output = gr.Plot(label="")

        # Trigger map on filter changes
        tier_filter.change(
            fn=build_map,
            inputs=[tier_filter, score_slider],
            outputs=map_output
        )
        score_slider.change(
            fn=build_map,
            inputs=[tier_filter, score_slider],
            outputs=map_output
        )

        # Load initial map on startup
        demo.load(
            fn=build_map,
            inputs=[tier_filter, score_slider],
            outputs=map_output
        )

    # ── Tab 2: Tract Narrative ───────────────────────────────────────────────
    with gr.Tab("📋 Tract Narrative"):
        gr.Markdown("""
        Enter any census tract GEOID to get a plain-English breakdown of its transit situation.
        **Tip:** GEOIDs are 11 digits. Miami-Dade tracts start with `12086`.
        You can find tract IDs on the map hover tooltips.
        """)

        with gr.Row():
            tract_input = gr.Textbox(
                label="Census Tract GEOID",
                placeholder="e.g.  12086010100",
                scale=2
            )
            tract_btn = gr.Button("Generate Narrative", variant="primary", scale=1)

        narrative_output = gr.Markdown(
            value="Enter a tract ID above and click **Generate Narrative**."
        )

        tract_btn.click(
            fn=generate_narrative,
            inputs=tract_input,
            outputs=narrative_output
        )
        # Also trigger on Enter key
        tract_input.submit(
            fn=generate_narrative,
            inputs=tract_input,
            outputs=narrative_output
        )

    # ── Tab 3: What-If Simulator ─────────────────────────────────────────────
    with gr.Tab("🔧 What-If Simulator"):
        gr.Markdown("""
        Adjust the **4 most impactful service levers** (from SHAP analysis) and see
        how the predicted deficit score changes. All other features are held at their
        Miami-Dade median values.

        | Feature | SHAP Importance |
        |---------|----------------|
        | Peak AM Frequency | 41% |
        | Weekend/Weekday Ratio | 24% |
        | Early AM Frequency | 7% |
        | Rail Trip Share | ~5% |
        """)

        with gr.Row():
            with gr.Column():
                s_headway = gr.Slider(
                    5, 90, value=30, step=5,
                    label="Peak AM Headway (minutes between buses)"
                )
                s_freq = gr.Slider(
                    0.5, 8.0, value=2.0, step=0.5,
                    label="Peak AM Frequency (trips per hour)"
                )
            with gr.Column():
                s_weekend = gr.Slider(
                    0.0, 1.5, value=0.7, step=0.05,
                    label="Weekend/Weekday Ratio (1.0 = same service)"
                )
                s_rail = gr.Slider(
                    0.0, 1.0, value=0.1, step=0.05,
                    label="Rail Trip Share"
                )

        sim_btn = gr.Button("▶ Simulate", variant="primary")
        sim_output = gr.Markdown(value="Adjust sliders and click **Simulate** to see predictions.")

        sim_btn.click(
            fn=whatif_simulate,
            inputs=[s_headway, s_freq, s_weekend, s_rail],
            outputs=sim_output
        )

    # ── Tab 4: Data Table ────────────────────────────────────────────────────
    with gr.Tab("📊 Data Table"):
        gr.Markdown("Sortable table of all tracts. Click column headers to sort.")

        display_cols = [
            'tract_geoid', 'risk_tier', 'deficit_predicted',
            'poverty_rate_pct', 'hh_no_vehicle_pct',
            'stop_count', 'route_count',
            'headway_peak_am_min', 'weekend_weekday_ratio',
            'freq_peak_am_tph', 'equity_tier'
        ]
        available_cols = [c for c in display_cols if c in df.columns]
        table_df = df[available_cols].sort_values('deficit_predicted', ascending=False).round(3)
        table_df['risk_tier'] = table_df['risk_tier'].astype(str)

        gr.Dataframe(
            value=table_df,
            interactive=False,
            wrap=False,
        )

    gr.Markdown("""
    ---
    *Data sources: GTFS (MDT schedules), ACS Census 2019–2023, ArcGIS isochrones.*
    *Model: XGBoost — Test R² 0.808, CV R² 0.821 ± 0.022.*
    """)


In [ ]:
# ── 9. Launch ─────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    demo.launch(share=True)   # share=True for Colab; remove for local

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://786f93eab695944055.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
